In [20]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from transliterate import translit

In [21]:
data = pd.read_csv("vacancys.csv")
data.sample(3)

,id,title,category,company,format,level,salary
2758,33491,Java Developer,development,Yandex Cloud,гибрид,middle,~ от 261 200 ₽
3745,32321,product manager,management,Kolesa Group,гибрид Алматы,middle,~ от 195 600 ₽
7654,26792,laravel developer,development,NDA,удалённо,senior,от 320 000 ₽


Для получения карточки вакансии требуется `id`, `category` (уже есть в датасете) и название вакансии записанное в латиннице. Например: 
> `https://hirehi.ru/analytics/biznes-analitik-36472`
- `id` = 36472
- `category` = analytics
- `title` = biznes-analitik

Часть названий записаны на русском, для трансляции в английский используется модуль [transliterate](https://docs-python.ru/packages/modul-transliterate-python/).

In [22]:
text = "Бизнес аналитик"
text = text.lower().strip()
text = translit(text, "ru", reversed=False)
text

'бизнес аналитик'

In [27]:
def get_skills(id, title, category):
    title = title.lower().strip()
    title = translit(title, "ru", reversed=True).replace(" ", "-")
    url = f"https://hirehi.ru/{category}/{title}-{id}"
    try:
        response = requests.get(url)
        page = BeautifulSoup(response.text, "html")
        tags_required = page.find_all("a", class_="job-tag")
        skills = []
        for tag in tags_required:
            skill = tag.text.lower().strip()
            skills.append(skill)
        return skills
    except Exception as e:
        print(f"Exception on {url}")
        return []

Получаем навыки из карточки вакасии: 

In [28]:
data["skills"] = data.apply(lambda x: get_skills(x.id, x.title, x.category), axis=1)

In [30]:
data["skills"].value_counts()

skills
[]                                                                                                                                                    417
[java, spring, j2ee, json, xml, postgresql, maven, git, ansible, jenkins, liquibase, docker, openshift]                                                 2
[aqa, python, javascript, java, api, ci/cd, gitlab ci, jenkins, микросервисы, docker, kubernetes, grafana, kibana]                                      2
[linux, bash, python, ansible, openstack, ceph, ci/cd, dns, elk, prometheus, git, kubernetes]                                                           2
[javascript, html5, css3, react, next.js, typescript, webpack, git, docker, kubernetes, nginx, scss, vue]                                               2
                                                                                                                                                     ... 
[sql, agile, scrum, product backlog, юнит-экономика, воронки, бизнес-

In [31]:
data.to_csv("vacancys_final.csv")